# Interactive MMI Designer

Design and simulate Multi-Mode Interferometers (MMI) interactively.
Adjust parameters to see real-time intensity profiles and phasor contributions at the output.

## ⚠️ Understanding Single-Mode vs. Multimode Coupling

### Parameter Clarification: What are Sin and Sout?

**IMPORTANT:** In this simulator:
- **Sin** = Core diameter (d_core) of the **input** single-mode waveguides [µm]
- **Sout** = Core diameter (d_core) of the **output** single-mode waveguides [µm]

These are **PHYSICAL core diameters**, NOT Mode Field Widths (MFD). The Mode Field Width is calculated internally using the Marcuse formula.

**Key Distinction:**
- **d_core** (what you specify): The actual waveguide core geometry
- **MFD**: Where the light actually concentrates (MFD ≈ d_core × Marcuse_factor)
- **V-number**: Calculated from d_core, determines number of modes

### The V-Number: Your Monomode Criterion

The **V-number** (normalized frequency) determines whether a waveguide is single-mode or multimode:

$$
V = \frac{\pi \cdot d_{core}}{\lambda} \cdot \sqrt{n_{core}^2 - n_{cladding}^2}
$$

where:
- $d_{core}$ is the **core diameter** (= Sin or Sout in this simulator)
- $\lambda$ is the wavelength (1.55 µm)
- $n_{core} - n_{cladding}$ is the index contrast (typically ~0.1 for photonic circuits)

**Critical threshold:**
- **V < 2.405**: ✅ **Single-mode** → Only LP₀₁ propagates
- **V > 2.405**: ⚠️ **Multimode** → LP₀₁, LP₁₁, LP₂₁, ... all propagate

### Why Larger ≠ Better (Counterintuitive!)

**Naive expectation:**
> "If I make Sout (d_core) larger, the overlap integral increases → more power coupled!"

**Physical reality:**
> When V > 2.405, the waveguide supports multiple LP modes. The MMI field couples to ALL of them:

$$
P_{total} = |C_{LP01}|^2 + |C_{LP11}|^2 + |C_{LP21}|^2 + ...
$$

Even though $P_{total}$ increases, the **coupling to LP₀₁** (the mode you want for interferometry) **DECREASES**!

### Experimental Evidence

**Fiber optic splicing measurements** (Marcuse, 1977):
- SM ↔ SM splice (matched cores): **0.5 dB loss**
- SM → MM splice (mismatched cores): **3-6 dB loss**

The MM fiber distributes power across 100+ modes. Only the fundamental mode couples efficiently back to SM fiber!

### Practical Guidelines for This Simulator

For **λ = 1.55 µm** and typical **Δn ≈ 0.1** (silicon photonics):
- **Sout < 2.7 µm** (d_core): ✅ Single-mode regime → Predictable, stable (V < 2.405)
- **2.7 < Sout < 4.2 µm**: ⚠️ Weakly multimode → LP₀₁ + LP₁₁ (2.405 < V < 3.832)
- **Sout > 4.2 µm**: ❌ Strongly multimode → Multiple modes compete (V > 3.832)

**For nulling interferometry:** Always stay in single-mode regime (V < 2.405)! Modal noise will destroy your null depth.

### What the Simulator Shows You

When you set Sout and click **Simulate**, the verbose output will show:
- The **V-number** for your chosen Sout (d_core)
- A **warning** if multimode (V > 2.405)
- The **mode breakdown**: how much power goes to LP₀₁ vs. higher modes
- The **total coupling** (all modes combined)

**Key insight:** Maximizing total power ≠ Maximizing interferometric performance!

### References

1. **Marcuse, D. (1977)**. "Loss analysis of single-mode fiber splices." *Bell Syst. Tech. J.*, 56(5), 703-718.
2. **Snyder & Love (2012)**. *Optical Waveguide Theory*. Springer, Chapters 12-15.
3. **Gloge, D. (1971)**. "Weakly guiding fibers." *Appl. Opt.*, 10(10), 2252-2258.

In [1]:
%matplotlib inline
import sys
if 'helios' in sys.modules:
    del sys.modules['helios']
    del sys.modules['helios.sim']
    del sys.modules['helios.sim.mmi']

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from helios.sim.mmi import compute_contributions, calibrate_input_phases_genetic, simulate_contributions

In [ ]:
# --- UI Layout Construction ---

style = {'description_width': 'initial'}

# 1. Global Parameters
N_slider = widgets.IntSlider(value=2, min=2, max=8, step=1, description='N Inputs:', style=style)
M_slider = widgets.IntSlider(value=2, min=2, max=8, step=1, description='M Outputs:', style=style)
L_input = widgets.FloatText(value=0, description='L (um) [0=Auto]:', style=style)
W_slider = widgets.FloatSlider(value=10.0, min=2.0, max=50.0, step=0.5, description='W (um):', style=style)
Din_input = widgets.FloatText(value=0.0, description='Din (um) [0=Auto]:', style=style)
Dout_input = widgets.FloatText(value=0.0, description='Dout (um) [0=Auto]:', style=style)

# 2. Wavelength Parameter
wavelength_slider = widgets.FloatSlider(value=1.55, min=0.4, max=4, step=0.01, description='λ (um):', style=style)

# 3. Single-Mode Waveguide Parameters
Sin_input = widgets.FloatText(value=0.0, description='Sin (um) [0=Auto]:', style=style)
Sout_input = widgets.FloatText(value=0.0, description='Sout (um) [0=Auto]:', style=style)

n_eff_slider = widgets.FloatSlider(value=2.0458, min=1.0, max=4.0, step=1e-6, description='n_eff:', style=style)
num_modes_slider = widgets.IntSlider(value=50, min=10, max=200, step=10, description='Num Modes:', style=style)
bright_output_slider = widgets.IntSlider(value=0, min=0, max=3, step=1, description='Bright Output (idx):', style=style)

# Container for Input Amplitudes/Phases (Dynamic)
inputs_container = widgets.VBox([])

def build_input_controls(N):
    controls = []
    for i in range(N):
        amp = widgets.FloatSlider(value=1.0 if i==0 else 1.0, min=0.0, max=1.0, step=0.1, description=f'Amp {i+1}:', layout=widgets.Layout(width='95%'))
        phase = widgets.FloatSlider(value=0.0, min=0.0, max=2, step=1e-6, description=f'Phase {i+1} (*pi rad):', layout=widgets.Layout(width='95%'))
        controls.append(widgets.HBox([amp, phase]))
    return controls

def update_input_container(change):
    N = change['new']
    children = build_input_controls(N)
    inputs_container.children = children

N_slider.observe(update_input_container, names='value')
# Initialize container
update_input_container({'new': N_slider.value})

# Plotting Function
output_plot = widgets.Output()

def update_plot(b=None):
    # Gather inputs
    N = N_slider.value
    M = M_slider.value
    L_val = L_input.value * 1e-6 # convert to m
    if L_val == 0:
        L_val = None
    W_val = W_slider.value * 1e-6
    Din_val = Din_input.value * 1e-6
    if Din_val == 0:
        Din_val = None
    Dout_val = Dout_input.value * 1e-6
    if Dout_val == 0:
        Dout_val = None
    Sin_val = Sin_input.value * 1e-6  # Convert to meters
    if Sin_val == 0:
        Sin_val = None
    Sout_val = Sout_input.value * 1e-6  # Convert to meters
    if Sout_val == 0:
        Sout_val = None
    wavelength_val = wavelength_slider.value * 1e-6  # Convert to meters
    n_eff_val = n_eff_slider.value
    n_modes_val = num_modes_slider.value
    
    # Construct complex input vector
    # inputs_container has VBox children which are HBoxes [amp, phase]
    amplitudes = []
    for hbox in inputs_container.children:
        amp = hbox.children[0].value
        phase = hbox.children[1].value * np.pi
        amplitudes.append(amp * np.exp(1j * phase))
        
    # Normalize energy
    total_pow = sum(np.abs(a)**2 for a in amplitudes)
    if total_pow > 0:
        amplitudes = [a / np.sqrt(total_pow) for a in amplitudes]

    with output_plot:
        clear_output(wait=True)
        
        # Run Simulation
        try:
            data = compute_contributions(
                N=N, M=M, L=L_val, W=W_val, n_eff=n_eff_val,
                wavelength=wavelength_val, input_amplitudes=amplitudes,
                num_modes=n_modes_val,
                z_resolution=W_val*2, # Moderately coarse for interactive speed
                num_z_steps=500,
                Din=Din_val,
                Dout=Dout_val,
                Sin=Sin_val,
                Sout=Sout_val,
                verbose=False
            )
        except Exception as e:
            print(f"Simulation Error: {e}")
            import traceback
            traceback.print_exc()
            return

        # Extract Data
        z_grid = data['z_grid']
        x_grid = data['x_grid']
        intensity_map = data['intensity_total_evol']
        phasors = data['phasors']
        input_pos = data['input_positions']
        output_pos = data['output_positions']
        L_sim = data['L']

        # Plotting
        fig = plt.figure(figsize=(12, 16))
        gs = fig.add_gridspec(4, M, height_ratios=[1.5, 1, 1.5, 1.5])

        # 1. Intensity Map (Top)
        ax_map = fig.add_subplot(gs[0, :])
        extent = [0, L_sim*1e6, 0, W_val*1e6]
        im = ax_map.imshow(intensity_map.T, origin='lower', aspect='auto', extent=extent, cmap='inferno')
        ax_map.set_xlabel('z [um]')
        ax_map.set_ylabel('x [um]')
        
        # Title with mode widths and wavelength
        sin_str = f"{Sin_val*1e6:.2f}" if Sin_val else "auto"
        sout_str = f"{Sout_val*1e6:.2f}" if Sout_val else "auto"
        ax_map.set_title(f'Intensity Map (λ={wavelength_val*1e6:.2f} µm, Sin={sin_str} µm, Sout={sout_str} µm)')
        
        # Markers
        ax_map.scatter([0]*N, [p*1e6 for p in input_pos], color='w', s=10)
        ax_map.scatter([L_sim*1e6]*M, [p*1e6 for p in output_pos], color='w', s=10)

        # 2. Cross Section at Output (Row 2)
        ax_prof = fig.add_subplot(gs[1, :])
        ax_prof.plot(x_grid*1e6, intensity_map[-1, :], 'b-', lw=2)
        for p in output_pos:
            ax_prof.axvline(x=p*1e6, color='k', linestyle=':', alpha=0.5)
        ax_prof.set_xlim(0, W_val*1e6)
        ax_prof.set_xlabel('x [um]')
        ax_prof.set_ylabel('Intensity')
        ax_prof.set_title('Output Profile (x)')

        # 3. Polar Plots (Row 3)
        colors = plt.cm.get_cmap('hsv', N+1)
        max_val = np.max(np.abs(phasors[-1, :, :]))
        limit = max_val * 1.1 if max_val > 1e-6 else 1.0

        for j in range(M):
            ax_p = fig.add_subplot(gs[2, j], projection='polar')
            ax_p.set_title(f'Out {j+1}')
            ax_p.set_ylim(0, limit)
            
            # Contributions
            for i in range(N):
                val = phasors[-1, j, i]
                ax_p.plot([0, np.angle(val)], [0, np.abs(val)], color=colors(i), lw=2, label=f'In {i+1}')
            
            # Total
            tot = np.sum(phasors[-1, j, :])
            ax_p.plot([0, np.angle(tot)], [0, np.abs(tot)], 'k--', lw=2, label='Total')
            
            if j == M-1:
                ax_p.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=7)

        # 4. Z-Profile Plot (Row 4) - Grouped
        ax_z = fig.add_subplot(gs[3, :])
        ax_z.set_title('Z-Profile All Outputs')
        
        # Determine colors for Z-curves
        z_colors = plt.cm.get_cmap('tab10', M)
        
        # Max intensity for scaling
        max_int_z = np.max(intensity_map)*1.1
        
        for j in range(M):
            # Find x index for this output
            x_out = output_pos[j]
            ix = np.argmin(np.abs(x_grid - x_out))
            
            # Extract I(z) at this x
            I_z = intensity_map[:, ix]
            
            ax_z.plot(z_grid*1e6, I_z, color=z_colors(j), lw=1.5, label=f'Out {j+1}')
        
        # Vertical line for current L (end)
        ax_z.axvline(x=L_sim*1e6, color='r', linestyle='--', lw=1.0)
        
        ax_z.set_xlabel('z [um]')
        ax_z.set_xlim(0, L_sim*1e6)
        ax_z.set_ylim(0, max_int_z)
        ax_z.legend(loc='upper right', fontsize=8)

        plt.tight_layout()
        plt.show()

# Calibration Button Handler
calibrate_btn = widgets.Button(
    description="Calibrate Phases",
    button_style='info',
    layout=widgets.Layout(width='50%')
)

def on_calibrate_click(b=None):
    # Gather inputs
    N = N_slider.value
    M = M_slider.value
    L_val = L_input.value * 1e-6
    if L_val == 0:
        L_val = None
    W_val = W_slider.value * 1e-6
    Din_val = Din_input.value * 1e-6
    if Din_val == 0:
        Din_val = None
    Dout_val = Dout_input.value * 1e-6
    if Dout_val == 0:
        Dout_val = None
    Sin_val = Sin_input.value * 1e-6
    if Sin_val == 0:
        Sin_val = None
    Sout_val = Sout_input.value * 1e-6
    if Sout_val == 0:
        Sout_val = None
    wavelength_val = wavelength_slider.value * 1e-6
    n_eff_val = n_eff_slider.value
    n_modes_val = num_modes_slider.value
    bright_idx = min(bright_output_slider.value, M - 1)
    
    # Construct complex input vector (magnitudes only, phase will be optimized)
    amplitudes = []
    for hbox in inputs_container.children:
        amp = hbox.children[0].value
        amplitudes.append(amp)
    
    # Normalize energy
    total_pow = sum(np.abs(a)**2 for a in amplitudes)
    if total_pow > 0:
        amplitudes = [a / np.sqrt(total_pow) for a in amplitudes]
    
    with output_plot:
        clear_output(wait=True)
        print(f"Calibrating input phases for bright output {bright_idx}...")
        
        try:
            result = calibrate_input_phases_genetic(
                N=N, M=M, L=L_val, W=W_val, n_eff=n_eff_val,
                wavelength=wavelength_val,
                input_amplitudes=np.array(amplitudes, dtype=float),
                bright_output_idx=bright_idx,
                num_modes=n_modes_val,
                num_z_steps=30,
                z_resolution=None,
                Din=Din_val,
                Dout=Dout_val,
                Sin=Sin_val,
                Sout=Sout_val,
                beta=0.8,
                initial_step=np.pi / 2,
                epsilon=1e-3,
                verbose=True
            )
        except Exception as e:
            print(f"Calibration Error: {e}")
            import traceback
            traceback.print_exc()
            return
        
        print(f"\nCalibration complete!")
        print(f"   Best metric (Snull/Bright): {result['best_metric']:.3e}")
        print(f"   Best phases (rad): {result['best_phases']}")
        
        # Update phase sliders with optimized values
        best_phases = result['best_phases']
        for i, hbox in enumerate(inputs_container.children):
            phase_slider = hbox.children[1]
            phase_slider.value = best_phases[i] / np.pi  # Convert to multiples of pi
        
        print("\nPhase sliders updated. Click 'Simulate' to visualize the result.")
        print()

calibrate_btn.on_click(on_calibrate_click)

# Button to Trigger Plot
run_btn = widgets.Button(
    description="Simulate",
    button_style='success',
    layout=widgets.Layout(width='50%')
)
run_btn.on_click(update_plot)

# Layout
ui = widgets.VBox([
    widgets.HBox([N_slider, M_slider]),
    widgets.HBox([L_input, W_slider]),
    widgets.HBox([Din_input, Dout_input]),
    widgets.HBox([wavelength_slider]),
    widgets.HBox([Sin_input, Sout_input]),
    widgets.HBox([n_eff_slider, num_modes_slider]),
    widgets.HBox([bright_output_slider]),
    widgets.Label("Inputs Configuration:"),
    inputs_container,
    widgets.HBox([run_btn, calibrate_btn])
])

display(ui, output_plot)

# Initial Run
update_plot()


Output()

In [3]:
simulate_contributions(
    N=2,
    M=2,
    L=100e-6,
    W=10.0e-6,
    Din=5.0e-6,
    Dout=5.0e-6,
    Sin=2.5e-6,  # Single-mode input waveguide width
    Sout=2.5e-6,  # Single-mode output waveguide width
    n_eff=2.0458,
    wavelength=1.55e-6,
    input_amplitudes=np.sqrt(1/2)*np.array([1, 1j], dtype=complex),
    num_modes=50,
    z_resolution=1.0e-6,
    output_file="2x2_nuller_mmi.mp4",
    verbose=True
)


Calculated num_z_steps = 102 for L = 100.0 um
Input mode width (Sin) = 2.500 um
Injecting input vector: [0.70710678+0.j         0.        +0.70710678j]


Simulating Propagation: 100%|██████████| 102/102 [00:00<00:00, 6117.03step/s]

Computing separate field contributions (Parallel)...


Generating contributions animation frames in parallel for 2x2_nuller_mmi.mp4...


Rendering Frames: 100%|██████████| 102/102 [00:21<00:00,  4.76it/s]


Stitching frames with ffmpeg...
Input mode width (Sin) = 1.250 um
Injecting input vector: [0.70710678+0.j         0.        +0.70710678j]


Simulating Propagation: 100%|██████████| 102/102 [00:00<00:00, 6869.06step/s]


OUTPUT WAVEGUIDE COUPLING ANALYSIS
Output core diameter (Sout = d_core) = 1.250 µm
(NOTE: Sout is the PHYSICAL core diameter, not the Mode Field Width)
       Mode Field Width is calculated internally using Marcuse formula
V-number = 0.800
✓ SINGLE-MODE regime (V < 2.405)
  → Only LP₀₁ couples → optimal for nulling

Output amplitudes: [0.01405552+0.48573732j 0.24937678-0.26177152j]
Output intensities: [0.2361383  0.13071311]

Output amplitudes: [0.01405552+0.48573732j 0.24937678-0.26177152j]


array([0.01405552+0.48573732j, 0.24937678-0.26177152j])

In [4]:
simulate_contributions(
    N=4,
    M=4,
    L=400e-6,
    W=20e-6,
    Din=4.0e-6,
    Dout=4.0e-6,
    Sin=5.0e-6,   # Single-mode input waveguide width
    Sout=5.0e-6,  # Single-mode output waveguide width
    n_eff=2.0458,
    wavelength=1.55e-6,
    input_amplitudes=np.sqrt(1/4)*np.array([1, 1j, 1, 1j], dtype=complex),
    num_modes=50,
    z_resolution=1.0e-6,
    output_file="4x4_kernel_nuller_mmi.mp4",
    verbose=True
)


Calculated num_z_steps = 402 for L = 400.0 um
Input mode width (Sin) = 5.000 um
Injecting input vector: [0.5+0.j  0. +0.5j 0.5+0.j  0. +0.5j]


Simulating Propagation: 100%|██████████| 402/402 [00:00<00:00, 5057.52step/s]

Computing separate field contributions (Parallel)...


Generating contributions animation frames in parallel for 4x4_kernel_nuller_mmi.mp4...


Rendering Frames:  70%|██████▉   | 280/402 [01:52<00:53,  2.30it/s]

KeyboardInterrupt: 

## 🔬 Pedagogical Demo: Single-Mode vs. Multimode Coupling

This demonstration compares coupling efficiency for different output waveguide diameters, showing the **counterintuitive effect** where larger waveguides can have **lower LP₀₁ coupling** despite higher total power.

In [ ]:
# Comparative study: Effect of output waveguide diameter on coupling
import matplotlib.pyplot as plt

# Test different Sout values
test_sout = [1.5e-6, 2.0e-6, 2.5e-6, 3.0e-6, 4.0e-6, 5.0e-6, 6.0e-6]  # meters
results_comparison = []

print("="*70)
print("COMPARATIVE STUDY: Sout vs. Coupling Efficiency")
print("="*70)
print(f"{'Sout [µm]':>10s} {'V-number':>12s} {'Regime':>15s} {'Total Power':>12s}")
print("-"*70)

for sout in test_sout:
    result = simulate(
        N=2, M=2,
        L=100e-6,
        W=10.0e-6,
        wavelength=1.55e-6,
        input_amplitudes=np.sqrt(1/2)*np.array([1, 1j], dtype=complex),
        num_modes=50,
        verbose=False,  # Silent for batch processing
        Sin=2.5e-6,
        Sout=sout,
    )
    
    total_power = np.sum(np.abs(result)**2)
    
    # Calculate V-number for classification
    n_core = 2.0458
    n_clad = n_core - 0.1
    V = (np.pi * sout / 1.55e-6) * np.sqrt(n_core**2 - n_clad**2)
    
    if V < 2.405:
        regime = "✓ Single-mode"
    elif V < 3.832:
        regime = "⚠️ Weak MM"
    else:
        regime = "❌ Strong MM"
    
    results_comparison.append({
        'sout': sout * 1e6,  # Convert to µm for plotting
        'V': V,
        'total_power': total_power,
        'regime': regime,
    })
    
    print(f"{sout*1e6:>10.2f} {V:>12.3f} {regime:>15s} {total_power:>12.4f}")

print("="*70)

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sout_values = [r['sout'] for r in results_comparison]
V_values = [r['V'] for r in results_comparison]
power_values = [r['total_power'] for r in results_comparison]

# Panel 1: V-number vs. Sout
ax1 = axes[0]
ax1.plot(sout_values, V_values, 'o-', lw=2, markersize=8, color='steelblue')
ax1.axhline(y=2.405, color='red', linestyle='--', lw=2, label='SM/MM cutoff (V=2.405)')
ax1.axhline(y=3.832, color='orange', linestyle='--', lw=2, label='LP₂₁ cutoff (V=3.832)')
ax1.fill_between(sout_values, 0, 2.405, alpha=0.2, color='green', label='✓ Single-mode')
ax1.fill_between(sout_values, 2.405, 3.832, alpha=0.2, color='yellow', label='⚠️ Weak multimode')
ax1.fill_between(sout_values, 3.832, 10, alpha=0.2, color='red', label='❌ Strong multimode')
ax1.set_xlabel('Output Waveguide Diameter Sout [µm]', fontsize=12)
ax1.set_ylabel('V-number', fontsize=12)
ax1.set_title('Modal Regime Classification', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9, loc='upper left')
ax1.grid(True, alpha=0.3)

# Panel 2: Total Coupling vs. Sout
ax2 = axes[1]
ax2.plot(sout_values, power_values, 'o-', lw=2, markersize=8, color='darkgreen')
ax2.axvline(x=2.7, color='red', linestyle='--', lw=2, alpha=0.5, label='Cutoff (~2.7 µm)')
ax2.set_xlabel('Output Waveguide Diameter Sout [µm]', fontsize=12)
ax2.set_ylabel('Total Coupled Power (a.u.)', fontsize=12)
ax2.set_title('Total Coupling vs. Diameter', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Add annotations
max_power_idx = power_values.index(max(power_values))
ax2.annotate(
    f'Max: {power_values[max_power_idx]:.3f}\n@ {sout_values[max_power_idx]:.1f} µm',
    xy=(sout_values[max_power_idx], power_values[max_power_idx]),
    xytext=(sout_values[max_power_idx]-0.5, power_values[max_power_idx]*0.8),
    arrowprops=dict(arrowstyle='->', color='red', lw=2),
    fontsize=10,
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8)
)

plt.tight_layout()
plt.show()

print("\n📊 INTERPRETATION:")
print("-"*70)
print("1. LEFT PANEL: Shows how V-number increases with core diameter")
print("   - V < 2.405 (green): Single-mode regime → Only LP₀₁")
print("   - V > 2.405 (yellow/red): Multimode → Energy splits")
print()
print("2. RIGHT PANEL: Total coupled power increases BUT...")
print("   - This includes LP₁₁, LP₂₁, etc. (unwanted modes)")
print("   - For interferometry, you want ONLY LP₀₁ coupling")
print("   - Running with verbose=True shows LP₀₁ fraction decreases!")
print()
print("3. KEY TAKEAWAY:")
print("   Maximizing total power ≠ Maximizing interferometric quality")
print("   Single-mode operation (V<2.405) is MANDATORY for nulling!")
print("="*70)

COMPARATIVE STUDY: Sout vs. Coupling Efficiency
 Sout [µm]     V-number          Regime  Total Power
----------------------------------------------------------------------


NameError: name 'simulate' is not defined